# DataRetail 360


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.utils.io import capture_output

current = Path.cwd().resolve()
ROOT = next((candidate for candidate in (current, *current.parents)
             if (candidate / "src" / "pipeline.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError(
        "No encuentro la raíz de DataRetail360_Pipeline. Abre el notebook "
        "desde la carpeta del proyecto o una de sus subcarpetas."
    )
sys.path.insert(0, str(ROOT))

SOURCE_PATH = ROOT / "data" / "online_retail_muestra_2000.csv"
DB_PATH = ROOT / "dataretail360_notebook.db"
SCHEMA_PATH = ROOT / "db" / "schema.sql"
assert SOURCE_PATH.exists(), f"No se encuentra la muestra: {SOURCE_PATH}"
print("Raíz del proyecto localizada correctamente.")
print(f"Fuente: {SOURCE_PATH.name}")
print(f"Filas de muestra: {len(pd.read_csv(SOURCE_PATH)):,}")

Raíz del proyecto localizada correctamente.
Fuente: online_retail_muestra_2000.csv
Filas de muestra: 2,000


In [2]:
from src.pipeline import execute_pipeline

# Se captura el log de consola para mostrar aquí el resumen final del job.
with capture_output():
    resultado = execute_pipeline(
        source_path=SOURCE_PATH,
        db_path=DB_PATH,
        schema_path=SCHEMA_PATH,
        batch_size=500,
        max_rows=2000,
    )

pd.DataFrame([{
    "Arquitectura": resultado["arquitectura"],
    "Estado": resultado["estado"],
    "Filas cargadas en Bronze": resultado["total_leidos"],
    "Silver válidas": resultado["total_validos"],
    "Rechazadas": resultado["total_rechazados"],
    "Hechos cargados en Gold": resultado["total_hechos"],
    "Bronze persistida antes de transformar": resultado["bronze_persistida_antes_de_transformar"],
    "Gold construida desde Silver persistida": resultado["gold_construida_desde_silver_persistida"],
    "Clientes no identificados conservados": resultado["filas_cliente_no_identificado"],
    "Duración (ms)": resultado["duracion_ms"],
}])

,Arquitectura,Estado,Filas cargadas en Bronze,Silver válidas,Rechazadas,Hechos cargados en Gold,Bronze persistida antes de transformar,Gold construida desde Silver persistida,Clientes no identificados conservados,Duración (ms)
0,ELT (Bronze-first) con SQLite,OK,2000,1992,8,1992,True,True,484,564.67


## Indicadores y controles SQL



In [3]:
from sqlalchemy import text
from src.load import make_engine

engine = make_engine(DB_PATH)
try:
    kpis = pd.read_sql_query("SELECT * FROM vw_kpis_ventas", engine)
    display(kpis)

    conteos = pd.read_sql_query(
        """SELECT
            (SELECT COUNT(*) FROM bronze_transacciones) AS bronze,
            (SELECT COUNT(*) FROM silver_transacciones) AS silver_validas,
            (SELECT COUNT(*) FROM silver_rechazos) AS rechazos,
            (SELECT COUNT(*) FROM fact_ventas) AS hechos,
            (SELECT COUNT(*) FROM silver_transacciones WHERE customer_identified=0) AS sin_customerid_conservadas,
            (SELECT COUNT(*) FROM dim_fecha) AS fechas,
            (SELECT COUNT(*) FROM dim_producto) AS productos,
            (SELECT COUNT(*) FROM dim_cliente WHERE tipo_scd='SCD2') AS filas_cliente_scd2,
            (SELECT COUNT(*) FROM dim_geografia) AS sucursales,
            (SELECT COUNT(DISTINCT region) FROM dim_geografia) AS regiones,
            (SELECT COUNT(DISTINCT pais) FROM dim_geografia) AS paises_distintos""",
        engine,
    )
    display(conteos)

    ventas_region = pd.read_sql_query("SELECT * FROM vw_ventas_por_region ORDER BY venta_neta DESC", engine)
    display(ventas_region)

    ventas_sucursal = pd.read_sql_query("SELECT * FROM vw_ventas_por_sucursal ORDER BY venta_neta DESC", engine)
    display(ventas_sucursal)

    rechazos = pd.read_sql_query(
        "SELECT rejection_reason, COUNT(*) AS filas FROM silver_rechazos GROUP BY rejection_reason ORDER BY filas DESC",
        engine,
    )
    display(rechazos)
finally:
    engine.dispose()

,lineas_validas,lineas_venta,lineas_devolucion,venta_bruta,monto_devoluciones,venta_neta,unidades_netas
0,1992,1952,40,35734.57,702.3,35032.27,19630


,bronze,silver_validas,rechazos,hechos,sin_customerid_conservadas,fechas,productos,filas_cliente_scd2,sucursales,regiones,paises_distintos
0,2000,1992,8,1992,484,305,1155,1233,6,3,1


,region,venta_bruta,devoluciones,venta_neta,unidades_netas,lineas_transaccion
0,Región Metropolitana de Santiago,25358.34,656.12,24702.22,13052,1375
1,Región de Valparaíso,5559.19,33.78,5525.41,3752,323
2,Región de O'Higgins,4817.04,12.40,4804.64,2826,294


,sucursal_codigo,sucursal,comuna,region,pais,venta_bruta,devoluciones,venta_neta,unidades_netas,lineas_transaccion
0,RM02,Sucursal Providencia,Providencia,Región Metropolitana de Santiago,Chile,7552.92,127.07,7425.85,3373,336
1,RM04,Sucursal Las Condes,Las Condes,Región Metropolitana de Santiago,Chile,6163.87,25.80,6138.07,2854,340
2,RM03,Sucursal Maipú,Maipú,Región Metropolitana de Santiago,Chile,5761.17,45.10,5716.07,2645,356
3,VA01,Sucursal Valparaíso,Valparaíso,Región de Valparaíso,Chile,5559.19,33.78,5525.41,3752,323
4,RM01,Sucursal Santiago Centro,Santiago,Región Metropolitana de Santiago,Chile,5880.38,458.15,5422.23,4180,343
5,OH01,Sucursal Rancagua,Rancagua,Región de O'Higgins,Chile,4817.04,12.40,4804.64,2826,294


,rejection_reason,filas
0,descripcion_faltante,5
1,precio_invalido,3


## Control final de integridad y alcance Chile

La consulta de `PRAGMA foreign_key_check` debe devolver una lista vacía. Además, la cantidad de hechos debe coincidir con Silver válida, Bronze debe ser igual a Silver válida más rechazos y no deben existir países diferentes de Chile en los datos persistidos.

In [4]:
engine = make_engine(DB_PATH)
try:
    with engine.connect() as conn:
        foreign_key_errors = conn.exec_driver_sql("PRAGMA foreign_key_check").fetchall()
        counts = conn.execute(text("""
            SELECT
                (SELECT COUNT(*) FROM bronze_transacciones) AS bronze,
                (SELECT COUNT(*) FROM silver_transacciones) AS silver_validas,
                (SELECT COUNT(*) FROM silver_rechazos) AS rechazos,
                (SELECT COUNT(*) FROM fact_ventas) AS hechos
        """)).one()
        countries = conn.execute(text("""SELECT
            (SELECT COUNT(*) FROM bronze_transacciones WHERE country_scope <> 'Chile') +
            (SELECT COUNT(*) FROM silver_transacciones WHERE country <> 'Chile') +
            (SELECT COUNT(*) FROM silver_rechazos WHERE country IS NOT NULL AND country <> 'Chile') +
            (SELECT COUNT(*) FROM dim_cliente WHERE pais <> 'Chile') +
            (SELECT COUNT(*) FROM dim_geografia WHERE pais <> 'Chile')""")).scalar_one()
        branches = conn.execute(text("SELECT COUNT(*) FROM dim_geografia")).scalar_one()
        region_counts = dict(conn.execute(text("SELECT region, COUNT(*) FROM dim_geografia GROUP BY region")).fetchall())
    print(f"Bronze = {counts[0]:,}; Silver = {counts[1]:,}; rechazos = {counts[2]:,}; Gold hechos = {counts[3]:,}")
    print(f"Errores de clave foránea: {len(foreign_key_errors)}")
    print(f"Registros fuera de Chile: {countries}")
    print(f"Sucursales modeladas: {branches}")
    print(f"Distribución por región: {region_counts}")
    assert not foreign_key_errors, foreign_key_errors
    assert counts[0] == counts[1] + counts[2], "Bronze debe ser Silver válida + rechazos"
    assert counts[1] == counts[3], "Silver válida debe coincidir con los hechos de Gold"
    assert countries == 0, "Se encontró un país fuera de Chile en los datos persistidos"
    assert branches == 6, "La dimensión debe incluir seis sucursales"
    assert region_counts == {
        "Región Metropolitana de Santiago": 4,
        "Región de O'Higgins": 1,
        "Región de Valparaíso": 1,
    }
    print("Validación ELT + alcance Chile: OK")
finally:
    engine.dispose()

Bronze = 2,000; Silver = 1,992; rechazos = 8; Gold hechos = 1,992
Errores de clave foránea: 0
Registros fuera de Chile: 0
Sucursales modeladas: 6
Distribución por región: {'Región Metropolitana de Santiago': 4, "Región de O'Higgins": 1, 'Región de Valparaíso': 1}
Validación ELT + alcance Chile: OK
